In [1]:
import re
import logging
from pathlib import Path
import pandas as pd
import pdfplumber

logging.basicConfig(level=logging.INFO, format="%(levelname)s | %(message)s")
logger = logging.getLogger(__name__)

In [2]:
# PDF_PATH = "../data/BOFA_072025_0504.pdf"
PDF_PATH = "../data/Chase_Sapphire_20251217-1333.pdf"
Capital_one_path = "../data/Capital_One_102025_2952.pdf"
BOFA_path = "../data/BOFA_092025_0504.pdf"
# PDF_PATH = "../data/Marcus_debit.PDF"
# PDF_PATH = '../data/Chase_College_20260325-8585.pdf'


`extract_text` — pull raw text from every PDF page

In [3]:
def extract_text(pdf_path: str) -> str:
    """Extract all text from a PDF using pdfplumber."""
    pages_text = []
    with pdfplumber.open(pdf_path) as pdf:
        for page in pdf.pages:
            text = page.extract_text() or ""
            pages_text.append(text)
    return "\n".join(pages_text)

# --- test ---
text = extract_text(PDF_PATH)
print(text[:2000])

MMaannaaggee yyoouurr aaccccoouunntt oonnlliinnee aatt:: Customer Service: MMoobbiillee:: DDoowwnnllooaadd tthhee
www.chase.com/cardhelp 1-800-493-3319 CChhaassee MMoobbiillee®® aapppp ttooddaayy
SCENARIO-1D
New Balance
January 2026
ULTIMATE REWARDS®
S M T W T F S
M$i6nim2u.m1 P9ayment Due
28 29 30 31 1 2 3 S Pr U evi M ous M po A ints R ba Y lance 3,534
4 5 6 7 8 9 10 P$a4ym0e.n0t D0ue Date + + 5 3 x x p p o o in in ts ts o o n n C d h in a i s n e g Travel 1,2 2 9 1 2 5
11 12 13 14 15 16 17
+ 2x points on other travel 1,553
18 19 20 21 22 23 24 01/14/26 + 3x points on online grocery 12
+ 1x points on all other purchases 664
25 26 27 28 29 30 31
1 2 3 4 5 6 7
Total points available for
LLatae Ptaeym enPt Waaryninmg:ent Warning: If we do not receive your minimum payment Lreeardn meomre apbotuito yonur rewards and start redeeming tod7ay,. 2Vi7sit0
by the date listed above, you may have to pay a late fee of up to Chase Ultimate Rewards® at www.UltimateRewards.com
$40.00 and your APRs may

`classify_account_type` — weighted regex scoring to detect credit / checking / savings

In [4]:
_ACCOUNT_SIGNALS: dict[str, list[tuple[re.Pattern, int]]] = {
    "credit": [
        (re.compile(r"minimum\s+payment\s+due",                  re.I), 3),
        (re.compile(r"credit\s+limit",                           re.I), 3),
        (re.compile(r"available\s+credit",                       re.I), 3),
        (re.compile(r"cash\s+advance",                           re.I), 3),
        (re.compile(r"purchase\s+apr",                           re.I), 3),
        (re.compile(r"payments.{0,10}credits.{0,10}adjustments", re.I), 2),
        (re.compile(r"rewards?\s+points?",                       re.I), 2),
        (re.compile(r"statement\s+balance",                      re.I), 1),
    ],
    "checking": [
        (re.compile(r"checks?\s+paid",                           re.I), 3),
        (re.compile(r"checking\s+account",                       re.I), 3),
        (re.compile(r"debit\s+card\s+purchases?",                re.I), 2),
        (re.compile(r"\boverdraft\b",                            re.I), 2),
        (re.compile(r"deposits?\s+and\s+additions",              re.I), 2),
        (re.compile(r"atm\s+withdrawal",                         re.I), 1),
        (re.compile(r"direct\s+deposit",                         re.I), 1),
    ],
    "savings": [
        (re.compile(r"online\s+savings",                         re.I), 3),
        (re.compile(r"high.yield\s+savings",                     re.I), 3),
        (re.compile(r"savings\s+account",                        re.I), 3),
        (re.compile(r"annual\s+percentage\s+yield",              re.I), 2),
        (re.compile(r"money\s+market",                           re.I), 2),
        (re.compile(r"interest\s+earned",                        re.I), 1),
        (re.compile(r"interest\s+(paid|credited)",               re.I), 1),
    ],
}


def classify_account_type(text: str) -> str | None:
    """Classify account type by scoring weighted regex signal hits across the full statement text."""
    scores = {account_type: 0 for account_type in _ACCOUNT_SIGNALS}
    for account_type, signals in _ACCOUNT_SIGNALS.items():
        for pattern, weight in signals:
            if pattern.search(text):
                scores[account_type] += weight

    best_type, best_score = max(scores.items(), key=lambda x: x[1])
    return best_type if best_score > 0 else None

# --- test ---
account_type = classify_account_type(text)
print("account_type:", account_type)

account_type: credit


`extract_last_four` — pull last 4 digits of account number from the statement header

In [5]:
_LAST_FOUR_RE = re.compile(
    r'(?:'
    r'ending\s+in\s+(\d{4})'
    r'|account\s*(?:number|no\.?|#)[:\s]+[ \d*xX-]*?(\d{4})\b'
    r'|\*{2,}(\d{4})\b'
    r'|[xX]{2,}(\d{4})\b'
    r'|(?:checking|savings)\s+\d*(\d{4})\b'
    r')',
    re.I,
)


def extract_last_four(text: str) -> str | None:
    """Return the last 4 digits of the account number from the statement header, or None."""
    account_match = _LAST_FOUR_RE.search(text[:3000])
    # print(account_match.groups())
    if account_match:
        return next(capture for capture in account_match.groups() if capture is not None)
    return None

# --- test ---
last_four = extract_last_four(text)
print("last_four:", last_four)

last_four: 1333


`extract_card_name` — identify the card/account product name from the header

In [6]:
_MARCUS_ACCOUNT_RE = re.compile(r'AccountName\s+([A-Za-z]+)', re.I)

_CARD_NAME_SIGNALS: list[tuple[re.Pattern, str]] = [
    # Chase credit
    (re.compile(r'sapphire\s+reserve',          re.I), "Chase Sapphire Reserve"),
    (re.compile(r'sapphire\s+preferred',        re.I), "Chase Sapphire Preferred"),
    (re.compile(r'sapphire',                    re.I), "Chase Sapphire"),
    (re.compile(r'freedom\s+unlimited',         re.I), "Chase Freedom Unlimited"),
    (re.compile(r'freedom\s+flex',              re.I), "Chase Freedom Flex"),
    (re.compile(r'freedom\s+rise',              re.I), "Chase Freedom Rise"),
    (re.compile(r'freedom',                     re.I), "Chase Freedom"),
    # Chase checking
    (re.compile(r'college\s+checking',          re.I), "Chase College Checking"),
    (re.compile(r'sapphire\s+checking',         re.I), "Chase Sapphire Checking"),
    (re.compile(r'premier\s+plus\s+checking',   re.I), "Chase Premier Plus Checking"),
    (re.compile(r'total\s+checking',            re.I), "Chase Total Checking"),
    # Capital One
    (re.compile(r'venture\s*one',               re.I), "Capital One VentureOne"),
    (re.compile(r'venture\s*x',                 re.I), "Capital One Venture X"),
    (re.compile(r'venture',                     re.I), "Capital One Venture"),
    (re.compile(r'quicksilver\s*one',           re.I), "Capital One QuicksilverOne"),
    (re.compile(r'quicksilver',                 re.I), "Capital One Quicksilver"),
    (re.compile(r'savor\s*one',                 re.I), "Capital One SavorOne"),
    (re.compile(r'savor',                       re.I), "Capital One Savor"),
    (re.compile(r'spark',                       re.I), "Capital One Spark"),
    # Bank of America
    (re.compile(r'customized\s+cash\s+rewards', re.I), "Bank of America Customized Cash Rewards"),
    (re.compile(r'unlimited\s+cash\s+rewards',  re.I), "Bank of America Unlimited Cash Rewards"),
    (re.compile(r'premium\s+rewards',           re.I), "Bank of America Premium Rewards"),
    (re.compile(r'travel\s+rewards',            re.I), "Bank of America Travel Rewards"),
    (re.compile(r'cash\s+rewards',              re.I), "Bank of America Cash Rewards"),
    (re.compile(r'visa\s+signature',            re.I), "Bank of America Visa Signature"),
    # Wells Fargo
    (re.compile(r'active\s+cash',               re.I), "Wells Fargo Active Cash"),
    (re.compile(r'autograph',                   re.I), "Wells Fargo Autograph"),
    (re.compile(r'reflect',                     re.I), "Wells Fargo Reflect"),
    # American Express
    (re.compile(r'platinum\s+card',             re.I), "Amex Platinum"),
    (re.compile(r'gold\s+card',                 re.I), "Amex Gold"),
    (re.compile(r'blue\s+cash\s+preferred',     re.I), "Amex Blue Cash Preferred"),
    (re.compile(r'blue\s+cash\s+everyday',      re.I), "Amex Blue Cash Everyday"),
    # Citi
    (re.compile(r'double\s+cash',               re.I), "Citi Double Cash"),
    (re.compile(r'custom\s+cash',               re.I), "Citi Custom Cash"),
    (re.compile(r'strata\s+premier',            re.I), "Citi Strata Premier"),
    # Discover
    (re.compile(r'discover\s+it',               re.I), "Discover It"),
]


def extract_card_name(text: str) -> str:
    """Return the card or account product name from the statement header."""
    header = text[:3000]

    marcus_match = _MARCUS_ACCOUNT_RE.search(header)
    if marcus_match:
        raw = marcus_match.group(1)
        name = re.sub(r'(?<=[a-z])(?=[A-Z])', ' ', raw)
        return f"Marcus {name}"

    best_pos, best_name = len(header) + 1, None
    for pattern, name in _CARD_NAME_SIGNALS:
        match = pattern.search(header)
        if match and match.start() < best_pos:
            best_pos, best_name = match.start(), name

    if best_name is None:
        return "Unknown Account"

    return best_name

# --- test ---
card_name = extract_card_name(text)
print("card_name:", card_name)

card_name: Chase Sapphire Preferred


`extract_statement_period` — parse the billing period dates from the header

In [7]:

_DATE_PAT = (
    r'\d{1,2}/\d{1,2}/\d{2,4}'
    r'|(?:Jan(?:uary)?|Feb(?:ruary)?|Mar(?:ch)?|Apr(?:il)?|May|Jun(?:e)?'
    r'|Jul(?:y)?|Aug(?:ust)?|Sep(?:tember)?|Oct(?:ober)?|Nov(?:ember)?|Dec(?:ember)?)'
    r'\s+\d{1,2},?\s+\d{4}'
)

# Month Day without year — for "August 7 - September 6, 2025" 
_DATE_NO_YEAR_PAT = (
    r'(?:Jan(?:uary)?|Feb(?:ruary)?|Mar(?:ch)?|Apr(?:il)?|May|Jun(?:e)?'
    r'|Jul(?:y)?|Aug(?:ust)?|Sep(?:tember)?|Oct(?:ober)?|Nov(?:ember)?|Dec(?:ember)?)'
    r'\s+\d{1,2},?'
)

_PERIOD_RE = re.compile(rf'({_DATE_PAT})\s*(?:through|to|[-–])\s*({_DATE_PAT})', re.I)
_PERIOD_SHARED_YEAR_RE = re.compile(rf'({_DATE_NO_YEAR_PAT})\s*[-–]\s*({_DATE_PAT})', re.I)


def extract_statement_period(text: str) -> dict:
    """Return {'period_start': 'YYYY-MM-DD', 'period_end': 'YYYY-MM-DD'} or Nones."""
    period_match = _PERIOD_RE.search(text[:3000])
    if period_match:
        try:
            return {
                "period_start": pd.to_datetime(period_match.group(1)).strftime("%Y-%m-%d"),
                "period_end":   pd.to_datetime(period_match.group(2)).strftime("%Y-%m-%d"),
            }
        except Exception:
            pass

    # Fallback: "Month Day - Month Day, Year" where year only appears on the end date (BofA)
    shared_year_match = _PERIOD_SHARED_YEAR_RE.search(text[:3000])
    if shared_year_match:
        try:
            end_date = pd.to_datetime(shared_year_match.group(2))
            start_date = pd.to_datetime(f"{shared_year_match.group(1)} {end_date.year}")
            if start_date > end_date:
                start_date = start_date.replace(year=end_date.year - 1)
            return {
                "period_start": start_date.strftime("%Y-%m-%d"),
                "period_end":   end_date.strftime("%Y-%m-%d"),
            }
        except Exception:
            pass

    return {"period_start": None, "period_end": None}


# --- test ---
# df = extract_transactions(text)
# print(f"{len(df)} transactions found")
# df

`extract_transactions` — parse transaction rows into a DataFrame

In [8]:
_DATE_PATTERN = (
    r'\b\d{1,2}/\d{1,2}(?:/\d{2,4})?\b|'
    r'\b(?:Jan(?:uary)?|Feb(?:ruary)?|Mar(?:ch)?|Apr(?:il)?|'
    r'May|Jun(?:e)?|Jul(?:y)?|Aug(?:ust)?|Sep(?:tember)?|'
    r'Oct(?:ober)?|Nov(?:ember)?|Dec(?:ember)?)\s+\d{1,2}\b'
)
_AMOUNT_PATTERN = r'-?\$?\d{1,3}(?:,\d{3})*(?:\.\d{2})?'

_TRANSACTION_ROW = re.compile(
    rf'^\s*({_DATE_PATTERN})'
    rf'(?:\s+({_DATE_PATTERN}))?'
    rf'\s+(.*?)'
    rf'\s+({_AMOUNT_PATTERN})'
    rf'(?:\s+({_AMOUNT_PATTERN}))?'
    rf'\s*$',
    re.I,
)

# Detects "payments … credits" section headers for amount-negation only (not for filtering).
_NEGATIVE_SECTION_RE = re.compile(
    r'\bpayments?\b.*\bcredits?\b|\bcredits?\b.*\bpayments?\b',
    re.I,
)

# Applied globally to every parsed row — these phrases never appear in real merchant names.
# Filters CC payment confirmations regardless of which PDF section they came from.
_CC_PAYMENT_FILTER = re.compile(
    r'payment\s+thank\s+you'    # Chase:        "Payment Thank You-Mobile"
    r'|electronic\s+payment'    # BofA:         "BA ELECTRONIC PAYMENT"
    r'|\bpymt\b'                # Capital One:  "CAPITAL ONE AUTOPAY PYMT"
    r'|\bautopay\b',            # Capital One / others: "AUTOPAY"
    re.I,
)


def extract_transactions(bank_text: str) -> pd.DataFrame:
    """Parse transaction rows from extracted PDF text into a DataFrame."""
    transactions = []
    is_negative_section = False

    for line in bank_text.splitlines():
        line = " ".join(line.split())
        if not line:
            continue

        if _NEGATIVE_SECTION_RE.search(line) and not _TRANSACTION_ROW.match(line):
            is_negative_section = True
            continue
        elif "TRANSACTIONS" in line.upper() and not _TRANSACTION_ROW.match(line):
            is_negative_section = False
            continue

        match = _TRANSACTION_ROW.match(line)
        if match:
            trans_date, post_date, description, amount1, amount2 = match.groups()
            description = description.strip()

            if _CC_PAYMENT_FILTER.search(description):
                continue

            amount1_num = float(amount1.replace("$", "").replace(",", ""))
            amount2_num = float(amount2.replace("$", "").replace(",", "")) if amount2 else None

            if is_negative_section:
                amount1_num = -abs(amount1_num)
                if amount2_num is not None:
                    amount2_num = -abs(amount2_num)

            transactions.append({
                "trans_date":  trans_date,
                "description": description,
                "amount1":     amount1_num,
                "amount2":     amount2_num,
            })

    return pd.DataFrame(transactions)

# --- test ---
df = extract_transactions(text)
print(f"{len(df)} transactions found")
df

83 transactions found


,trans_date,description,amount1,amount2
0,11/28,HM Hennes Mauritz UK L London,-46.18,None
1,11/17,TIAN TIAN MARKET - CANARY LONDON,-53.06,None
2,11/17,CL *Chase Travel TRIPCHRG.COM VA,-42.81,None
3,11/16,VENICE TRANSPORT VENEZIA,-1.75,None
4,11/16,TFL TRAVEL CH TFL.GOV.UK/CP,-2.31,None
...,...,...,...,...
78,12/12,TFL TRAVEL CH TFL.GOV.UK/CP,-7.77,None
79,12/13,TFL TRAVEL CH TFL.GOV.UK/CP,-9.52,None
80,12/14,Amazon Fresh amazon.co.uk,-3.79,None
81,12/16,GETYOURGUIDE TICKETS 855-957-1272 NY,-29.15,None


`parse_pdf` — orchestrates all of the above into a single result dict

In [9]:
def parse_pdf(pdf_path: str) -> dict:
    text = extract_text(pdf_path)
    df = extract_transactions(text)
    period = extract_statement_period(text)
    account_type = classify_account_type(text)
    last_four = extract_last_four(text)
    card_name = extract_card_name(text)

    logger.info(
        "Parsed %s: %d transactions, period %s → %s, account_type %s, last_four %s, card_name %s",
        Path(pdf_path).name, len(df),
        period["period_start"], period["period_end"],
        account_type, last_four, card_name,
    )

    return {
        "period_start": period["period_start"],
        "period_end":   period["period_end"],
        "account_type": account_type,
        "last_four":    last_four,
        "card_name":    card_name,
        "transactions": df,
    }

# --- test ---
result = parse_pdf(PDF_PATH)
print({k: v for k, v in result.items() if k != "transactions"})
result["transactions"]

INFO | Parsed Chase_Sapphire_20251217-1333.pdf: 83 transactions, period 2025-11-18 → 2025-12-17, account_type credit, last_four 1333, card_name Chase Sapphire Preferred


{'period_start': '2025-11-18', 'period_end': '2025-12-17', 'account_type': 'credit', 'last_four': '1333', 'card_name': 'Chase Sapphire Preferred'}


,trans_date,description,amount1,amount2
0,11/28,HM Hennes Mauritz UK L London,-46.18,None
1,11/17,TIAN TIAN MARKET - CANARY LONDON,-53.06,None
2,11/17,CL *Chase Travel TRIPCHRG.COM VA,-42.81,None
3,11/16,VENICE TRANSPORT VENEZIA,-1.75,None
4,11/16,TFL TRAVEL CH TFL.GOV.UK/CP,-2.31,None
...,...,...,...,...
78,12/12,TFL TRAVEL CH TFL.GOV.UK/CP,-7.77,None
79,12/13,TFL TRAVEL CH TFL.GOV.UK/CP,-9.52,None
80,12/14,Amazon Fresh amazon.co.uk,-3.79,None
81,12/16,GETYOURGUIDE TICKETS 855-957-1272 NY,-29.15,None


In [10]:
parse_pdf(Capital_one_path)

INFO | Parsed Capital_One_102025_2952.pdf: 73 transactions, period 2025-09-20 → 2025-10-20, account_type credit, last_four 2952, card_name Capital One VentureOne


{'period_start': '2025-09-20',
 'period_end': '2025-10-20',
 'account_type': 'credit',
 'last_four': '2952',
 'card_name': 'Capital One VentureOne',
 'transactions':    trans_date                       description  amount1 amount2
 0      Oct 15           aliexpressSan MateoCA -    -7.89    None
 1      Oct 17   GETYOURGUIDE TICKETSLONDONGBR -  -109.79    None
 2      Oct 17   GETYOURGUIDE TICKETSLONDONGBR -   -34.41    None
 3      Sep 19        WWW.VOXI.CO.UKVODAFONE LTD    13.73    None
 4      Sep 21  SumUp *fresh meetcanning townGBR    15.59    None
 ..        ...                               ...      ...     ...
 68     Oct 18         SQ *BREAD AHEAD LTDLondon     6.06    None
 69     Oct 18      TIAN TIAN MARKET - CANLONDON    66.58    None
 70     Oct 18       ASDA SUPERSTOREISLE OF DOGS    18.38    None
 71     Oct 18        TFL TRAVEL CHTFL.GOV.UK/CP     3.91    None
 72     Oct 18               MARUGAME UDONLONDON    17.45    None
 
 [73 rows x 4 columns]}

In [11]:
parse_pdf(BOFA_path)

INFO | Parsed BOFA_092025_0504.pdf: 5 transactions, period 2025-09-07 → 2025-10-06, account_type credit, last_four 4400, card_name Bank of America Visa Signature


{'period_start': '2025-09-07',
 'period_end': '2025-10-06',
 'account_type': 'credit',
 'last_four': '4400',
 'card_name': 'Bank of America Visa Signature',
 'transactions':   trans_date                              description  amount1 amount2
 0      09/19                  TRAVEL CREDIT 9123 0504   -29.85    None
 1      10/06            INTEREST CHARGED ON PURCHASES    -0.00    None
 2      10/06    INTEREST CHARGED ON BALANCE TRANSFERS    -0.00    None
 3      10/06  INTEREST CHARGED ON DIR DEP&CHK CASHADV    -0.00    None
 4      10/06   INTEREST CHARGED ON BANK CASH ADVANCES    -0.00    None}

In [12]:
parse_pdf(PDF_PATH)

INFO | Parsed Chase_Sapphire_20251217-1333.pdf: 83 transactions, period 2025-11-18 → 2025-12-17, account_type credit, last_four 1333, card_name Chase Sapphire Preferred


{'period_start': '2025-11-18',
 'period_end': '2025-12-17',
 'account_type': 'credit',
 'last_four': '1333',
 'card_name': 'Chase Sapphire Preferred',
 'transactions':    trans_date                           description  amount1 amount2
 0       11/28         HM Hennes Mauritz UK L London   -46.18    None
 1       11/17      TIAN TIAN MARKET - CANARY LONDON   -53.06    None
 2       11/17      CL *Chase Travel TRIPCHRG.COM VA   -42.81    None
 3       11/16              VENICE TRANSPORT VENEZIA    -1.75    None
 4       11/16           TFL TRAVEL CH TFL.GOV.UK/CP    -2.31    None
 ..        ...                                   ...      ...     ...
 78      12/12           TFL TRAVEL CH TFL.GOV.UK/CP    -7.77    None
 79      12/13           TFL TRAVEL CH TFL.GOV.UK/CP    -9.52    None
 80      12/14             Amazon Fresh amazon.co.uk    -3.79    None
 81      12/16  GETYOURGUIDE TICKETS 855-957-1272 NY   -29.15    None
 82      12/01                 ANNUAL MEMBERSHIP FEE   -95.00  